# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s using `mlcroissant`.

In [ ]:
from pprint import pprint

# List available record sets by @id
print("Available Record Sets:")
record_sets = [rs['@id'] for rs in metadata._json_ld.get('recordSet', [])]
for rs in record_sets:
    print(f"- RecordSet @id: {rs}")

# For each record set, print fields and their @id
def get_fields_for_recordset(rs_id):
    # Find the recordset dict matching @id
    for rs in metadata._json_ld.get('recordSet', []):
        if rs['@id'] == rs_id:
            fields = rs.get('field', [])
            if isinstance(fields, dict):
                fields = [fields]
            return fields
    return []

for rs in record_sets:
    print(f"\nFields for RecordSet {rs}:")
    fields = get_fields_for_recordset(rs)
    for f in fields:
        if isinstance(f, dict):
            print(f"  - {f.get('@id', f)}")
        else:
            print(f"  - {f}")

## 3. Data Extraction
Load data from each record set into a `pandas` DataFrame for analysis. We reference each record set and field via their `@id` fields as shown above.

In [ ]:
# If there are multiple record sets, list their @ids found above
# If none found, attempt default (the most common case is "cr:RecordSet/0" or similar ID pattern)

if record_sets:
    used_record_sets = record_sets
    print(f"Using detected RecordSets: {used_record_sets}")
else:
    # Attempt fallback
    used_record_sets = ['cr:RecordSet/0']
    print("No record sets found in metadata, using fallback: 'cr:RecordSet/0'")

dataframes = {}
for rs_id in used_record_sets:
    try:
        print(f"\nLoading records from RecordSet {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Fields (columns) in {rs_id}:\n", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate filtering, normalization, and grouping using numeric and categorical fields. Please adjust the `numeric_field_id` and `group_field_id` using `@id` references from the previous code block.

In [ ]:
# Example: Use PHQ-9 score as a numeric field and gender as a group field if present

# Adjust these values to match actual @id values for your dataset
# You can examine dataframes[rs_id].columns to determine possible values
rs_id = used_record_sets[0] if used_record_sets else None
df = dataframes[rs_id]

# Try common possible ids for numeric mental health scores
possible_numeric_fields = ['phq9_score', 'PHQ9_total', 'phq-9', 'gad7_score', 'psq_score', 'total_score']
numeric_field = None
for field in possible_numeric_fields:
    if field in df.columns:
        numeric_field = field
        break
# If none found, just use the first numeric column
if not numeric_field:
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    numeric_field = numeric_cols[0] if numeric_cols else df.columns[0]

print(f"Using numeric field: {numeric_field}")

# Try to find a group field, e.g. gender or village
possible_group_fields = ['gender', 'village', 'level_of_education', 'marital_status', 'age_group']
group_field = None
for field in possible_group_fields:
    if field in df.columns:
        group_field = field
        break
print(f"Using group field: {group_field}")

# Filtering: e.g. PHQ-9 score > 10 (common severity cutoff)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by the group_field if present
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Mean {numeric_field} grouped by {group_field}:")
    display(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
df[numeric_field].dropna().hist(bins=20)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot of numeric_field by group_field if available
if group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f'{numeric_field} distribution by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. We reviewed metadata, record sets, and fields referenced by their `@id` values. We demonstrated basic exploratory analysis, including filtering, normalization, grouping, and visualizations of mental health score distributions.

- For best results, customize field identifiers to match the Croissant schema's actual `@id` values.
- This workflow can be adapted for further, in-depth analysis as needed.